# Clasificaci?n Escalamiento: 8 y 16 Qubits

Este notebook est? configurado para ajustar autom?ticamente la resoluci?n de la imagen (v?a PCA) al n?mero de qubits seleccionados.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# === CONFIGURACI?N ===
n_qubits = 16  # <--- CAMBIA AQU? A 8 O 16
# =====================

# 1. Carga de datos (D?gitos 0 y 1)
digits = datasets.load_digits()
mask = np.isin(digits.target, [0, 1])
X_raw = digits.data[mask]
y_raw = digits.target[mask]

# 2. PCA din?mico seg?n el n?mero de qubits
pca = PCA(n_components=n_qubits)
X_pca = pca.fit_transform(X_raw)

# Escalado
X_scaled = MinMaxScaler(feature_range=(0, 1)).fit_transform(X_pca)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_raw, test_size=0.2, random_state=42)

encoder = OneHotEncoder(sparse_output=False)
y_train_oh = encoder.fit_transform(y_train.reshape(-1, 1))
y_test_oh = encoder.transform(y_test.reshape(-1, 1))

print(f"Configurado para {n_qubits} qubits.")
print(f"Varianza explicada total: {np.sum(pca.explained_variance_ratio_)*100:.2f}%")

### Construcci?n del Circuito para {n_qubits} Qubits

In [ ]:
from qiskit.circuit.library import zz_feature_map, real_amplitudes

# Generar circuitos din?micamente
feature_map = zz_feature_map(n_qubits, reps=2)
ansatz = real_amplitudes(n_qubits, reps=1)

print(f"Feature Map y Ansatz creados para {n_qubits} qubits.")

### Entrenamiento con VQC

In [ ]:
from qiskit_machine_learning.algorithms import VQC
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler

vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=COBYLA(maxiter=100),
    sampler=StatevectorSampler()
)

print("Entrenando...")
vqc.fit(X_train, y_train_oh)
print("?Listo!")

### Evaluaci?n Final

In [ ]:
accuracy = vqc.score(X_test, y_test_oh)
print(f"Precisi?n final ({n_qubits} qubits): {accuracy * 100:.2f}%")